In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
from pyspark.sql import functions as F

df_systems = spark.table("lh_silver.dbo.silver_pvdaq_systems")

df_telemetry = spark.table("lh_silver.dbo.silver_pvdaq_telemetry")

print("Silver systems:", df_systems.count())
print("Silver telemetry:", df_telemetry.count())

df_telemetry.printSchema()

In [ ]:
df_telemetry = spark.table(
    "`uae-solar-dev`.lh_silver.dbo.silver_pvdaq_telemetry"
)

print("Total telemetry records:", df_telemetry.count())

df_telemetry.printSchema()

display(df_telemetry.limit(5))

In [ ]:
silver_path = (
    "abfss://ac09d151-f58f-40f3-bf13-a61a3c24bcdb"
    "@onelake.dfs.fabric.microsoft.com/"
    "f63acb2c-3844-4da7-8866-f3b7c155a001/"
    "Tables/silver_pvdaq_telemetry"
)

df_telemetry = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Telemetry records:", df_telemetry.count())

df_telemetry.printSchema()

display(df_telemetry.limit(5))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

time_window = Window.partitionBy("system_id").orderBy("measurement_timestamp")

df_intervals = (
    df_telemetry
    .withColumn(
        "previous_timestamp",
        F.lag("measurement_timestamp").over(time_window)
    )
    .withColumn(
        "interval_minutes",
        (
            F.unix_timestamp("measurement_timestamp")
            - F.unix_timestamp("previous_timestamp")
        ) / 60
    )
)

df_intervals.select(
    "system_id",
    "measurement_timestamp",
    "previous_timestamp",
    "interval_minutes",
    "ac_power_raw",
    "dc_power_raw"
).orderBy("measurement_timestamp").show(15, truncate=False)

df_intervals.agg(
    F.min("interval_minutes").alias("min_interval"),
    F.max("interval_minutes").alias("max_interval"),
    F.avg("interval_minutes").alias("avg_interval"),
    F.min("ac_power_raw").alias("min_ac_power"),
    F.max("ac_power_raw").alias("max_ac_power"),
    F.min("dc_power_raw").alias("min_dc_power"),
    F.max("dc_power_raw").alias("max_dc_power")
).show()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

time_window = Window.partitionBy("system_id").orderBy("measurement_timestamp")

df_intervals = (
    df_telemetry
    .withColumn(
        "previous_timestamp",
        F.lag("measurement_timestamp").over(time_window)
    )
    .withColumn(
        "interval_minutes",
        (
            F.unix_timestamp("measurement_timestamp")
            - F.unix_timestamp("previous_timestamp")
        ) / 60
    )
)

df_intervals.select(
    "system_id",
    "measurement_timestamp",
    "previous_timestamp",
    "interval_minutes",
    "ac_power_raw",
    "dc_power_raw"
).orderBy("measurement_timestamp").show(15, truncate=False)

df_intervals.agg(
    F.min("interval_minutes").alias("min_interval"),
    F.max("interval_minutes").alias("max_interval"),
    F.avg("interval_minutes").alias("avg_interval"),
    F.min("ac_power_raw").alias("min_ac_power"),
    F.max("ac_power_raw").alias("max_ac_power"),
    F.min("dc_power_raw").alias("min_dc_power"),
    F.max("dc_power_raw").alias("max_dc_power")
).show()

In [ ]:
from pyspark.sql import functions as F

df_fact_telemetry = (
    df_telemetry
    .select(
        "system_id",
        "measurement_timestamp",
        "measurement_date",
        "ac_power_raw",
        "dc_power_raw"
    )
    .withColumn(
        "ac_power_kw",
        F.col("ac_power_raw") / 1000.0
    )
    .withColumn(
        "dc_power_kw",
        F.col("dc_power_raw") / 1000.0
    )
    .withColumn(
        "power_unit_status",
        F.lit("ASSUMED_WATTS_PENDING_VERIFICATION")
    )
)

display(df_fact_telemetry.limit(10))

In [ ]:
import os

print("Default Lakehouse:")
print(os.path.realpath("/lakehouse/default"))

print("\nAvailable Lakehouse mounts:")
print(os.listdir("/lakehouse"))

In [ ]:
from pyspark.sql import functions as F

# ============================================================
# STEP 1 — ONELAKE CONFIGURATION
# ============================================================

workspace_id = "ac09d151-f58f-40f3-bf13-a61a3c24bcdb"

silver_lakehouse_id = "f63acb2c-3844-4da7-8866-f3b7c155a001"

silver_base_path = (
    f"abfss://{workspace_id}"
    f"@onelake.dfs.fabric.microsoft.com/"
    f"{silver_lakehouse_id}/Tables"
)

silver_telemetry_path = (
    f"{silver_base_path}/silver_pvdaq_telemetry"
)

silver_systems_path = (
    f"{silver_base_path}/dbo/silver_pvdaq_systems"
)

# ============================================================
# STEP 2 — READ SILVER DELTA TABLES
# ============================================================

print("Reading Silver telemetry...")

df_telemetry = (
    spark.read
    .format("delta")
    .load(silver_telemetry_path)
)

print("Telemetry records:", df_telemetry.count())

print("Reading Silver systems...")

df_systems = (
    spark.read
    .format("delta")
    .load(silver_systems_path)
)

print("Systems records:", df_systems.count())

# ============================================================
# STEP 3 — VALIDATE SILVER RECORD COUNTS
# ============================================================

assert df_telemetry.count() == 328, "Telemetry count mismatch"

assert df_systems.count() == 1858, "Systems count mismatch"

print("Silver validation successful.")

# ============================================================
# STEP 4 — CREATE GOLD DIM_PLANT
# ============================================================

df_dim_plant = (
    df_systems
    .select(
        F.col("system_id").alias("plant_id"),
        F.col("system_public_name").alias("plant_name"),
        F.col("site_location"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("dc_capacity_kW").alias("dc_capacity_kw"),
        F.col("tilt"),
        F.col("azimuth")
    )
    .dropDuplicates(["plant_id"])
)

# ============================================================
# STEP 5 — CREATE GOLD DIM_DATE
# ============================================================

date_bounds = df_telemetry.agg(
    F.min("measurement_date").alias("start_date"),
    F.max("measurement_date").alias("end_date")
).first()

df_dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                DATE '{date_bounds.start_date}',
                DATE '{date_bounds.end_date}',
                INTERVAL 1 DAY
            )
        ) AS date
        """
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
)

# ============================================================
# STEP 6 — CREATE GOLD FACT_SOLAR_TELEMETRY
# ============================================================

df_fact_telemetry = (
    df_telemetry
    .select(
        F.col("system_id").alias("plant_id"),
        "measurement_timestamp",
        "measurement_date",
        "ac_power_raw",
        "dc_power_raw"
    )
    .withColumn(
        "date_key",
        F.date_format(
            "measurement_date",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "ac_power_kw_provisional",
        F.col("ac_power_raw") / 1000.0
    )
    .withColumn(
        "dc_power_kw_provisional",
        F.col("dc_power_raw") / 1000.0
    )
    .withColumn(
        "power_unit_status",
        F.lit("ASSUMED_WATTS_PENDING_VERIFICATION")
    )
)

# ============================================================
# STEP 7 — VERIFY GOLD DATAFRAMES
# ============================================================

print("\nGOLD VALIDATION RESULTS")
print("-----------------------")

print("dim_plant:", df_dim_plant.count())
print("dim_date:", df_dim_date.count())
print("fact_solar_telemetry:", df_fact_telemetry.count())

assert df_dim_plant.count() == 1858
assert df_dim_date.count() == 7
assert df_fact_telemetry.count() == 328

print("\nGold DataFrames prepared successfully.")

# ============================================================
# STEP 8 — DISPLAY SAMPLE DATA
# ============================================================

print("\nDIM_PLANT")
display(df_dim_plant.limit(5))

print("\nDIM_DATE")
display(df_dim_date.orderBy("date").limit(7))

print("\nFACT_SOLAR_TELEMETRY")
display(df_fact_telemetry.limit(5))

In [ ]:
# ============================================================
# STEP 43 — SAVE AND VERIFY GOLD DELTA TABLES
# ============================================================

from delta.tables import DeltaTable

gold_tables = {
    "dim_plant": df_dim_plant,
    "dim_date": df_dim_date,
    "fact_solar_telemetry": df_fact_telemetry
}

# Save each DataFrame as a registered Gold Delta table.

for table_name, dataframe in gold_tables.items():

    full_table_name = f"dbo.{table_name}"

    if spark.catalog.tableExists(full_table_name):
        print(f"SKIPPED: {full_table_name} already exists.")
        continue

    (
        dataframe.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(full_table_name)
    )

    print(f"CREATED: {full_table_name}")

# ============================================================
# VERIFY GOLD TABLES
# ============================================================

print("\nGOLD TABLE VERIFICATION")
print("-----------------------")

expected_counts = {
    "dim_plant": 1858,
    "dim_date": 7,
    "fact_solar_telemetry": 328
}

for table_name, expected_count in expected_counts.items():

    full_table_name = f"dbo.{table_name}"

    actual_count = spark.table(full_table_name).count()

    print(
        f"{full_table_name}: "
        f"{actual_count} records"
    )

    assert actual_count == expected_count, (
        f"Validation failed for {full_table_name}"
    )

print("\nSUCCESS: All three Gold tables verified.")